# Homework 3 — Spark first steps

About 30–45 minutes. You practise everything from session 3: creating DataFrames, looking at them, `select`,
`F.col`, `withColumn`, functions like `F.round`, `F.lower`, `F.trim`, `F.length`, `F.lit`, and reading a CSV file.

Every exercise shows the expected output, so you can check yourself.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

### Exercise 1 — A DataFrame of playlists
Turn `playlists` into a DataFrame called `playlists_df`. Show it, print its schema and count the rows.

In [ ]:
playlists = [
    {"name": "  morning run ", "owner": "U0101", "n_tracks": 25, "total_ms": 5_430_000},
    {"name": "Focus", "owner": "U0102", "n_tracks": 40, "total_ms": 9_120_000},
    {"name": "road TRIP", "owner": "U0101", "n_tracks": 60, "total_ms": 13_800_000},
    {"name": "Sleep  ", "owner": "U0103", "n_tracks": 12, "total_ms": 3_600_000},
    {"name": "party", "owner": "U0104", "n_tracks": 80, "total_ms": 17_760_000},
]

# YOUR CODE HERE

playlists_df.show()
playlists_df.printSchema()
print(playlists_df.count(), "playlists")  # expected: 5 playlists

Which type did Spark choose for `n_tracks` and `total_ms`? Why are the columns in a different order than in `playlists`?

### Exercise 2 — Pick columns
Show only `name` and `n_tracks`, in that order.

```
+--------------+--------+
|          name|n_tracks|
+--------------+--------+
|  morning run |      25|
|         Focus|      40|
|     road TRIP|      60|
|       Sleep  |      12|
|         party|      80|
+--------------+--------+
```

In [ ]:
# YOUR CODE HERE

### Exercise 3 — Minutes
Build `playlists_min` from `playlists_df` with two new columns:
- `total_minutes`: `total_ms` / 60000, rounded to 1 decimal
- `avg_minutes`: `total_minutes` / `n_tracks`, rounded to 2 decimals (you can use a column you created one step before!)

Then show `name`, `n_tracks`, `total_minutes`, `avg_minutes`.

```
+--------------+--------+-------------+-----------+
|          name|n_tracks|total_minutes|avg_minutes|
+--------------+--------+-------------+-----------+
|  morning run |      25|         90.5|       3.62|
|         Focus|      40|        152.0|        3.8|
|     road TRIP|      60|        230.0|       3.83|
|       Sleep  |      12|         60.0|        5.0|
|         party|      80|        296.0|        3.7|
+--------------+--------+-------------+-----------+
```

In [ ]:
# YOUR CODE HERE

playlists_min.select("name", "n_tracks", "total_minutes", "avg_minutes").show()

### Exercise 4 — Clean the names
The playlist names have extra spaces and mixed casing. Add two columns to `playlists_df`:
- `name_clean`: the name without spaces at the ends, in lower case (nest two functions)
- `name_length`: the number of characters of the **clean** name

Store the result in `playlists_clean` and show `name`, `name_clean`, `name_length`.

```
+--------------+-----------+-----------+
|          name| name_clean|name_length|
+--------------+-----------+-----------+
|  morning run |morning run|         11|
|         Focus|      focus|          5|
|     road TRIP|  road trip|          9|
|       Sleep  |      sleep|          5|
|         party|      party|          5|
+--------------+-----------+-----------+
```

In [ ]:
# YOUR CODE HERE

playlists_clean.select("name", "name_clean", "name_length").show()

### Exercise 5 — The same value in every row
Select `name` and `owner`, then add a column `platform` with the text `"Wavely"` and a column `year` with the number `2026`.

```
+--------------+-----+--------+----+
|          name|owner|platform|year|
+--------------+-----+--------+----+
|  morning run |U0101|  Wavely|2026|
|         Focus|U0102|  Wavely|2026|
|     road TRIP|U0101|  Wavely|2026|
|       Sleep  |U0103|  Wavely|2026|
|         party|U0104|  Wavely|2026|
+--------------+-----+--------+----+
```

In [ ]:
# YOUR CODE HERE

### Exercise 6 — Read the artists
Read `DATA / "reference" / "artists.csv"` into `artists` with `header=True, inferSchema=True`. Print the schema and the number of rows.
Then show `artist_name`, `country`, a column `name_length` and a column `label` like `Taylor Swift (pop)` (hint: `F.concat` and `F.lit`).

```
+-------------+-------+-----------+-------------------+
|  artist_name|country|name_length|              label|
+-------------+-------+-----------+-------------------+
| Taylor Swift|     US|         12| Taylor Swift (pop)|
|     Dua Lipa|     GB|          8|     Dua Lipa (pop)|
|   The Weeknd|     CA|         10|   The Weeknd (pop)|
|Billie Eilish|     US|         13|Billie Eilish (pop)|
|   Ed Sheeran|     GB|         10|   Ed Sheeran (pop)|
+-------------+-------+-----------+-------------------+
only showing top 5 rows
```

In [ ]:
# YOUR CODE HERE

### Exercise 7 (Stretch) — Durations as `m:ss`
In session 2 you wrote `ms_to_text(ms)` in plain Python. Do the same for a whole column in Spark: read `tracks.csv`
(with `inferSchema=True`) and add a column `duration` like `3:57`.

Hints: `F.floor(x)` gives the whole part of a number, `%` gives the remainder, and
`F.format_string("%d:%02d", col1, col2)` works like an f-string with `{a}:{b:02d}`.

```
+----------------+-----------+--------+
|           title|duration_ms|duration|
+----------------+-----------+--------+
|Electric Mirrors|     237000|    3:57|
|Electric Skyline|     200000|    3:20|
|Endless Promises|     198000|    3:18|
|   Golden Dreams|     193000|    3:13|
|    Little Ocean|     186000|    3:06|
+----------------+-----------+--------+
only showing top 5 rows
```

In [ ]:
# YOUR CODE HERE